# Demo: Prophet

Prophet takes a different approach from ARIMA. Instead of thinking in terms of AR and MA orders, it decomposes the forecast into trend + seasonality + regressors. You don't specify (p,d,q) -- Prophet figures out the structure automatically. The tradeoff is less control, but for many business forecasting problems that's fine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date").asfreq("MS")
train = df.loc[:"2023-12-01"]
print(f"Train: {len(train)} months")

## Preparing Data for Prophet

Prophet wants a DataFrame with columns named `ds` (date) and `y` (target). Any extra regressors go in their own columns. It doesn't use a DatetimeIndex.

In [ ]:
prophet_df = train.reset_index().rename(columns={"date": "ds", "demand_mwh": "y"})
prophet_df["avg_temp_f"] = train["avg_temp_f"].values
prophet_df.head()

## Fitting Prophet

We tell it to use yearly seasonality, turn off weekly and daily (monthly data doesn't have those), and add temperature as a regressor.

In [ ]:
m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
m.add_regressor("avg_temp_f")
m.fit(prophet_df[["ds", "y", "avg_temp_f"]])

In [ ]:
# Build future dataframe with temperature
seasonal_temp = train.groupby(train.index.month)["avg_temp_f"].mean()
future = m.make_future_dataframe(periods=12, freq="MS")
future["avg_temp_f"] = list(train["avg_temp_f"].values) + [seasonal_temp[m] for m in range(1, 13)]

forecast = m.predict(future)
pred = forecast.iloc[-12:]

## Component Plot

This is Prophet's killer feature. It breaks the forecast into pieces so you can see what's driving it.

In [ ]:
m.plot_components(forecast)
plt.tight_layout()
plt.show()

The trend line shows the long-run direction. The yearly seasonality shows the annual cycle. And the temperature effect shows how demand responds to temperature independently of the seasonal pattern. You get all of this for free -- no manual specification.

## Comparing to ARIMA

In [ ]:
# Quick ARIMA baseline
arima = SARIMAX(train["demand_mwh"], order=(2, 1, 1)).fit(disp=False)
arima_fc = arima.get_forecast(12)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train.index, train["demand_mwh"].values, color="black", label="Historical")
ax.plot(arima_fc.predicted_mean.index, arima_fc.predicted_mean.values, color="tab:blue", linestyle="--", label="ARIMA")
ci = arima_fc.conf_int()
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="tab:blue", alpha=0.1)

pred_idx = pd.date_range("2024-01-01", periods=12, freq="MS")
ax.plot(pred_idx, pred["yhat"].values, color="tab:green", linestyle="--", label="Prophet")
ax.fill_between(pred_idx, pred["yhat_lower"].values, pred["yhat_upper"].values, color="tab:green", alpha=0.1)

ax.set_ylabel("Demand (MWh)")
ax.legend()
plt.tight_layout()
plt.show()

Prophet captures the seasonal shape that ARIMA misses entirely. The tradeoff: with ARIMA you control every parameter and can run formal diagnostics. With Prophet, you get sensible defaults and component plots but less insight into what the model is doing mathematically. Both have their place.